# Taller 1 - Gestión de Datos
## Grupo: Juan Camilo Guerrero, Maria Camila Rincón, Luis Alberto Banda, Juan Pablo Marin y Grabriel Camilo Velasquez
## 1. Datos

**Objetivo del análisis:** preparar y cargar las tablas del taller de forma reproducible, para poder explorarlas en las secciones siguientes.

**Descripción del dataset:** se encuentra en la carpeta `Data` del workspace y consta de tres archivos:

| Archivo | Formato | Contenido |
|---|---|---|
| `countries.csv` | CSV | Tabla de países con su código, latitud, longitud y nombre. |
| `diabetes.csv` | CSV | Tabla de mediciones clínicas de pacientes y un indicador de diabetes (`Outcome`). |
| `Currencies.json` | JSON | Tabla de monedas por país, con sus códigos y nombres. |

### 1.1 Librerías
Importamos las librerías que usaremos durante todo el taller.

In [1]:
# Librerías del taller
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

### 1.2 Archivos disponibles
Listamos los archivos de `Data` con su tamaño. Si alguno pesa menos de 1 KB, probablemente es un puntero de Git LFS (no el dato real) y detenemos la ejecución.

In [2]:
# Ubicamos la carpeta Data (el notebook vive en work/, Data está un nivel arriba)
RUTAS_POSIBLES = [Path("../Data"), Path("Data")]
DATA_DIR = next((r for r in RUTAS_POSIBLES if r.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontró la carpeta 'Data'. Si usas Docker, recuerda que solo "
        "./work está montada en el contenedor: monta también ./Data."
    )

# Listamos cada archivo (csv y json) con su tamaño
archivos = sorted(list(DATA_DIR.glob("*.csv")) + list(DATA_DIR.glob("*.json")))
for archivo in archivos:
    print(f"{archivo.name:<20} {archivo.stat().st_size / 1024:>8.1f} KB")

# Un archivo < 1 KB es casi seguro un puntero de Git LFS
pequenos = [a.name for a in archivos if a.stat().st_size < 1024]
if pequenos:
    raise SystemExit(
        f"Detenido: {pequenos} pesan menos de 1 KB (puntero de Git LFS). "
        "Ejecuta 'git lfs pull' para descargar los datos reales."
    )

Currencies.json          44.2 KB
countries.csv             8.5 KB
diabetes.csv             23.3 KB


### 1.3 Carga de tablas
Cargamos `countries`, `Currencies` y `diabetes`. Convertimos a fecha las columnas cuyo nombre contenga `date` o `timestamp`, o termine en `_at`, e imprimimos el `shape` de cada tabla.

In [3]:
def es_columna_fecha(nombre):
    n = str(nombre).lower()
    return "date" in n or "timestamp" in n or n.endswith("_at")

tablas = {
    "countries": pd.read_csv(DATA_DIR / "countries.csv"),
    "Currencies": pd.read_json(DATA_DIR / "Currencies.json"),
    "diabetes": pd.read_csv(DATA_DIR / "diabetes.csv"),
}

for nombre, df in tablas.items():
    cols_fecha = [c for c in df.columns if es_columna_fecha(c)]
    for c in cols_fecha:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    print(f"{nombre:<11} shape={df.shape}  columnas de fecha convertidas: {cols_fecha or 'ninguna'}")

countries, Currencies, diabetes = tablas["countries"], tablas["Currencies"], tablas["diabetes"]

countries   shape=(245, 4)  columnas de fecha convertidas: ninguna
Currencies  shape=(254, 5)  columnas de fecha convertidas: ninguna
diabetes    shape=(768, 9)  columnas de fecha convertidas: ninguna


## 2. Perfilamiento

Resumimos cada atributo de las tablas cargadas: tipo de almacenamiento, tipo conceptual, nulos, valores distintos y estadísticos básicos.

> **Importante:** la columna `tipo_conceptual_sugerido` sale de reglas simples (dtype, número de valores distintos y nombre de la columna). Es solo una **sugerencia**: el analista debe validarla.

In [4]:
def tipo_conceptual(serie):
    """Sugiere el tipo conceptual de una columna con reglas simples (validar manualmente)."""
    n = serie.dropna()
    if pd.api.types.is_datetime64_any_dtype(serie):
        return "fecha"
    if isinstance(serie.dtype, pd.CategoricalDtype) and serie.dtype.ordered:
        return "ordinal"
    # Identificador: todos los valores distintos (no flotantes) o nombre típico de código/id
    nombre = str(serie.name).lower()
    es_id_por_nombre = nombre == "id" or nombre.endswith("_id") or "code" in nombre
    if not pd.api.types.is_float_dtype(serie) and len(n) > 0 and (n.nunique() == len(n) or es_id_por_nombre):
        return "identificador"
    if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
        return "nominal"
    if n.nunique() <= 2:
        return "nominal"  # variable binaria
    return "discreta" if pd.api.types.is_integer_dtype(serie) else "continua"


def perfil(df, tabla):
    """Devuelve una fila de perfilamiento por cada atributo de df."""
    filas = []
    for col in df.columns:
        s = df[col]
        es_num = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
        es_fecha = pd.api.types.is_datetime64_any_dtype(s)
        moda = s.mode(dropna=True)
        filas.append({
            "atributo": col,
            "tabla": tabla,
            "dtype": str(s.dtype),
            "tipo_conceptual_sugerido": tipo_conceptual(s),
            "nulos": int(s.isna().sum()),
            "%_nulos": round(s.isna().mean() * 100, 2),
            "distintos": int(s.nunique(dropna=True)),
            "media": s.mean() if es_num else np.nan,
            "desv_est": s.std() if es_num else np.nan,
            "moda": moda.iloc[0] if len(moda) else np.nan,
            "minimo": s.min() if (es_num or es_fecha) else np.nan,
            "maximo": s.max() if (es_num or es_fecha) else np.nan,
        })
    return pd.DataFrame(filas)


# Aplicamos la función a todas las tablas cargadas y concatenamos
perfiles = pd.concat([perfil(df, nombre) for nombre, df in tablas.items()], ignore_index=True)
perfiles = perfiles.sort_values(["%_nulos", "tabla"], ascending=[False, True], ignore_index=True)
perfiles

,atributo,tabla,dtype,tipo_conceptual_sugerido,nulos,%_nulos,distintos,media,desv_est,moda,minimo,maximo
0,country,countries,object,identificador,1,0.41,244,NaN,NaN,AD,NaN,NaN
1,latitude,countries,float64,continua,1,0.41,244,16.253109,27.031206,-75.250973,-75.250973,77.553604
2,longitude,countries,float64,continua,1,0.41,244,13.294814,73.976477,-177.156097,-177.156097,179.414413
3,Country,Currencies,object,identificador,0,0.00,254,NaN,NaN,ALand Islands,NaN,NaN
4,Country code (Alpha-2),Currencies,object,identificador,0,0.00,254,NaN,NaN,AC,NaN,NaN
5,Country code (Alpha-3),Currencies,object,identificador,0,0.00,253,NaN,NaN,,NaN,NaN
6,Currency,Currencies,object,nominal,0,0.00,155,NaN,NaN,Euro,NaN,NaN
7,Currency code,Currencies,object,identificador,0,0.00,155,NaN,NaN,EUR,NaN,NaN
8,name,countries,object,identificador,0,0.00,245,NaN,NaN,Afghanistan,NaN,NaN
9,Pregnancies,diabetes,int64,discreta,0,0.00,17,3.845052,3.369578,1.0,0.000000,17.000000
